# Clase NN — Título de la clase

**Deep Learning para Cientistas Sociales**

> **La idea de hoy en una línea.** Acá va la intuición central de la clase,
> escrita como se la dirías a alguien en un pasillo. Sin fórmulas.

---

**Cuando termines esta clase vas a poder:**

1. (objetivo concreto, verificable, en infinitivo)
2. (otro)
3. (otro, y no más de cuatro)

**Lo que hace falta traer:** la clase anterior. Nada más.

**Cómo se usa este notebook.** Corré las celdas en orden, de arriba hacia abajo
(`Shift + Enter`). Donde diga `# TODO` te toca a vos: el notebook está pensado
para que escribas, no para que mires correr código ajeno. Si algo explota,
mirá `recursos/setup.md`, sección *si algo explota*.


## 0. Preparar el entorno

Esta celda es **idéntica en los diez notebooks del curso**. En Colab clona el
repo y deja todo a mano; en tu máquina no hace prácticamente nada porque ya
tenés los archivos. Corrémela y seguí de largo: no hay nada que entender acá.


In [ ]:
# =============================================================================
# BOOTSTRAP — la misma celda en todos los notebooks del curso. No la edites.
# En Colab: clona el repo del curso e instala lo que falte.
# En tu máquina: encuentra el repo subiendo carpetas y no instala nada.
# =============================================================================
import os
import pathlib
import subprocess
import sys

REPO = "https://github.com/johannperez/dlss"
RAMA = "claude/dl-course-social-scientists-279dd2"
CARPETA = "dlss"


def en_colab():
    """¿Estamos en Google Colab? Se lo preguntamos a Python, no lo adivinamos."""
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def buscar_raiz(desde=None):
    """Sube carpeta por carpeta hasta encontrar la que tiene `_shared/ssdl`."""
    punto = pathlib.Path(desde or os.getcwd()).resolve()
    for candidata in [punto, *punto.parents]:
        if (candidata / "_shared" / "ssdl").is_dir():
            return candidata
    return None


raiz = buscar_raiz()

if raiz is None and en_colab():
    if not pathlib.Path(CARPETA).exists():
        print("Bajando el repo del curso (una sola vez por sesión)...")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,
                        REPO, CARPETA], check=True)
    raiz = buscar_raiz(CARPETA)

if raiz is None:
    raise SystemExit(
        "No encuentro el repo del curso.\n"
        "· En Colab: revisá que la celda haya podido clonar (necesita internet).\n"
        "· En tu máquina: abrí este notebook desde adentro de la carpeta `dlss`."
    )

# Dependencias: en Colab casi todas vienen puestas. Instalamos sólo lo que falte.
for modulo, paquete in [("numpy", "numpy"), ("matplotlib", "matplotlib"),
                        ("torch", "torch")]:
    try:
        __import__(modulo)
    except ImportError:
        print(f"Instalando {paquete}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete],
                       check=True)

sys.path.insert(0, str(raiz / "_shared"))   # para poder hacer `import ssdl`
os.environ["DLSS_RAIZ"] = str(raiz)         # para que ssdl encuentre los datos

print(f"Listo. Repo del curso en: {raiz}")

### Imports y semilla

`ssdl` es el paquete del curso: la plomería que no queremos repetir en diez
notebooks (cargar datos, dibujar con los colores de siempre, el loop de
entrenamiento). No esconde nada importante — está en `_shared/ssdl/` y se puede
leer. La referencia completa está en `_shared/ssdl/README.md`.

Fijamos la semilla en 1983 **siempre**. No hace al modelo mejor: hace a tu
resultado reproducible, que es condición para poder afirmar algo.


In [ ]:
import numpy as np
import torch

from ssdl import datos, entrenamiento as ent, graficos as g, utils

utils.fijar_semilla(1983)          # la semilla del curso
utils.resumen_del_entorno()        # dónde estamos parados y con qué versiones

---

## 1. La idea

> Acá va la explicación en prosa: la intuición primero, la fórmula después (y a
> veces no va). Dos o tres párrafos como máximo. Si necesitás más, es que hay
> dos ideas y conviene partir la sección.
>
> Regla de la casa: cada concepto abstracto necesita una imagen mental
> concreta. Si no se te ocurre ninguna, el concepto todavía no está listo para
> entrar al notebook.

**La analogía de ciencias sociales.** (La comparación con algo que la audiencia
ya sabe hacer: una regresión, un análisis de contenido, una codificación con
dos codificadores y su kappa.)

**Lo que NO es.** (Desarmar acá la confusión típica, antes de que aparezca.)


---

## 2. Miralo

Antes de escribir una línea, mirá el fenómeno. Corré la celda y quedate un
minuto con el gráfico.

> **Apostá primero.** Antes de correr: ¿qué esperás ver? Sin la predicción
> fallida, un gráfico lindo es sólo un gráfico lindo.


In [ ]:
# Los datos que nos van a acompañar de la clase 1 a la 4: qué porcentaje de las
# palabras del discurso presidencial de apertura se dedicó a "seguridad",
# año por año. Datos reales (ver datos/README.md).
anios, seguridad = datos.serie_menciones("seguridad", desde=1984, hasta=2019)

print(f"{len(anios)} años, de {int(anios[0])} a {int(anios[-1])}")
print(f"El máximo fue en {int(anios[seguridad.argmax()])}: "
      f"{seguridad.max():.2f}% de las palabras del discurso.")

g.plot_serie(anios, seguridad, etiqueta="menciones a seguridad",
             titulo="Cuánto se habló de seguridad en la apertura de sesiones")

La visualización interactiva de esta clase está en `viz/nombre-de-la-viz.html`.
Abrila en el navegador (funciona sin internet) y movele los controles antes de
seguir: el notebook da por sentado que ya jugaste con ella.


---

## 3. Hacelo vos

Ahora con las manos. Primero lo hacemos **a mano**, y recién después usamos la
función que lo hace sola — esa es la regla de la casa número 3: una abstracción
sólo se permite después de haber hecho a mano lo que esconde.


In [ ]:
# --- 3.1 A mano -------------------------------------------------------------
# Un modelo con dos perillas: y = a * x + b. Movelas a ojo y mirá qué pasa.

# Arrancamos con una recta horizontal, deliberadamente mal puesta.
a = 0.000     # TODO: la INCLINACIÓN. Probá 0.002, 0.004, -0.003.
b = 0.10      # TODO: la ALTURA. Probá 0.15, 0.20. ¿Para qué lado mueve la recta?

def mi_recta(x):
    return a * x + b

# El error de UN caso es la distancia vertical entre el punto y la recta.
# La loss es el promedio de esos errores al cuadrado (por eso se llama MSE).
errores = seguridad - mi_recta(anios)
loss = np.mean(errores ** 2)

print(f"MSE = {loss:.5f}     (más chico es mejor)")
print(f"El caso peor ajustado es {int(anios[np.abs(errores).argmax()])}, "
      f"con un error de {np.abs(errores).max():.3f} puntos.")
print("\nAhora movele las perillas de arriba y volvé a correr la celda.")

g.plot_ajuste(anios, seguridad, modelo=mi_recta, loss=loss,
              xlabel="año", ylabel="% de palabras",
              etiqueta_modelo=f"mi recta (a={a}, b={b})",
              titulo="Ajustar a ojo: girar las perillas y mirar el número")

In [ ]:
# --- 3.2 Ahora que lo hiciste a mano, que lo haga PyTorch -------------------
# Es exactamente lo mismo, pero las perillas se mueven solas cuesta abajo.
# Centramos y escalamos el año: si no, `a` y `b` quedan tan correlacionadas
# que el descenso zigzaguea. (Volvemos sobre esto en la clase 3.)

x = torch.tensor((anios - anios.mean()) / 10, dtype=torch.float32).reshape(-1, 1)
y = torch.tensor(seguridad, dtype=torch.float32).reshape(-1, 1)

modelo = torch.nn.Linear(1, 1)        # dos perillas: un peso y un sesgo
print(f"El modelo tiene {utils.contar_parametros(modelo)} perillas.")

historial = ent.entrenar(modelo, x, y, lr=0.03, epocas=200, verboso=2)

In [ ]:
# ¿Qué encontró? Comparalo con lo que habías puesto a ojo.
prediccion = ent.predecir(modelo, x, como_numpy=True).ravel()

g.plot_ajuste(anios, seguridad, y_pred=prediccion,
              xlabel="año", ylabel="% de palabras",
              etiqueta_modelo="lo que encontró el descenso",
              titulo="La misma recta, pero las perillas las movió el gradiente")

historial.graficar(titulo="Cómo fue bajando la loss")

---

## 4. Ejercicio

**Consigna.** (Una sola, concreta, que se pueda resolver en 10 minutos y que
obligue a usar la idea de la clase, no a copiar la celda de arriba.)

**Pista.** (Una pista que destrabe sin resolver.)

**Cómo saber si te salió.** (El criterio explícito. Un `assert`, un número
esperado, una forma que el gráfico tiene que tener.)


In [ ]:
# TODO: tu solución acá.
#
# Ejercicio: repetí el ajuste con el tema "inflacion" en vez de "seguridad".
# ¿La recta ajusta igual de bien? ¿Por qué?

anios_inf, inflacion = datos.serie_menciones("inflacion", desde=1984, hasta=2019)

# TODO: ajustá una recta a esta serie y calculá su MSE.
# TODO: graficala con g.plot_ajuste y mirá los errores.
# TODO: escribí en la celda de abajo, en una oración, por qué una recta
#       no alcanza acá. (Esa respuesta es el gancho de la clase 2.)

g.plot_serie(anios_inf, inflacion, etiqueta="menciones a inflación",
             titulo="La serie del ejercicio: ¿le sirve una recta?")

**Tu respuesta:** _(escribí acá)_


---

## 5. Qué te llevás

**Las tres cosas de hoy:**

1. (La idea central, dicha como se la contarías a un colega en el pasillo.)
2. (La segunda.)
3. (La tercera, que suele ser la advertencia: cuándo esto te miente.)

**El vocabulario nuevo** (también está en `recursos/glosario.md`):

| término | en castellano llano |
|---|---|
| *loss* | un número que mide qué tan mal la estás dibujando |
| *(término)* | *(su explicación en una línea)* |

**Para el lunes, en tu propia investigación.** (La bajada concreta: qué de esto
podés usar mañana, y con qué recaudo.)

**Lo que queda prometido.** (La siembra: qué pregunta abrimos hoy que se paga
en una clase futura. Anotala, porque se cobra.)

---

### Para seguir

Las referencias de la clase están en `referencias.md`. Las de ciencias sociales
van primero: lo que importa no es el paper de ML, es el trabajo que muestra qué
se puede y qué no se puede afirmar con esto.
